# Quantum Phase Estimation of a Qubitized Hamiltonian

## Part I. The Hamiltonian

Quantum simulation of physical systems lies at the heart of many of the most interesting large scale fault-tolerant quantum algorithms. Domains such as materials science and chemistry immediately spring to mind, but the true potential of quantum simulation is limited only by our imaginations. Finance, fluid dynamics, optimization and partial differential equation solvers are a small subset of the broad range of applications that have been found thus far with their roots in quantum simulation.

At the core of these applications lies a deceptively simple idea: take a problem domain, map it to a physical system, solve the dynamics of that system on a quantum computer, then postprocess the output to reveal some interesting feature of the problem domain. It turns out that we can often find systems and mappings that permit a substantial quantum advantage over classical algorithms. We're going to take this idea and apply it to a toy model, starting from a mere sketch of the system we want to analyze and breaking it all the way down to a full gate implementation.

Our model is intentionally simple. Models corresponding to actual, real-world problems are much more complicated. Even so, many of the techniques we will discuss here have direct application to the more complicated models, giving you a toolkit that can be used to compile more useful applications. Moreover, throughout the kata we will promote and reinforce a test-driven approach to quantum computing, with substantial effort given to verifying the behavior and validating the output of the primitives we build. Embracing this approach will make it much easier to build robust, reliable quantum software.

This kata covers the following topics:

* The workflow of constructing a Hamiltonian based on the properties and behavior of a physical system
* Implementing and validating block encoding and qubitization iterate of a Hamiltonian
* Running quantum phase estimation on a qubitization iterate to discover the properties of a Hamiltonian
* Analyzing the resource requirements of a quantum program and using the results of this analysis to improve the algorithm

> This is an advanced kata, and it covers a lot of ground! Throughout the text, we include pointers to other, more introductory katas that will help you learn each topic starting from the basic concepts.

This kata consists of four parts that follow the typical flow of solving a quantum computing problem:

1. Part I (this notebook) walks you through the theoretical construction of the Hamiltonian we'll use as the example and its mathematical properties.
2. [Part II](./QubitizedHamiltonianQPE_2_BlockEncoding.ipynb) implements and validates the block encoding of this Hamiltonian as a Workbench program.
3. [Part III](./QubitizedHamiltonianQPE_3_QPE.ipynb) builds on top of part II, implementing a qubitization iterate based on the block encoding and running quantum phase estimation to reveal some numeric properties of our Hamiltonian.
4. [Part IV](./QubitizedHamiltonianQPE_4_QRE.ipynb) demonstrates basic resource analysis of our program, using Workbench QRE tools and algorithmic insight to help optimize our solution.
5. [Part V](./QubitizedHamiltonianQPE_5_LandscapeAnalysis.ipynb) walks you through a more advanced example of resource analysis of our program, showing how to plot the algorithm's resource costs as a function of its parameters and to analyze any irregularities in its behavior.

## Defining the Hamiltonian
For the remainder of this kata, we will work with a specific Hamiltonian. The Hamiltonian was chosen specifically to provide a nice example to block encode rather than for any useful application. Nevertheless, we will derive it as you would any physical system &mdash; that is, by starting from a picture rather than from any mathematics!

The system we're looking to analyze is a simple model of particles on a one dimensional chain with $N$ sites and periodic boundary conditions (i.e. we treat the site $N-1$ as being adjacent to site $0$, as if the ends of the chain are connected). The particles can occupy any of the $N$ sites. We will encode the chain in a quantum state such that site $k$ on the chain corresponds to basis state $\ket{k}$ in the state. This means that using $n$ qubits we can encode $N = 2^n$ sites. We can illustrate this with a diagram, showing empty sites as white circles and an occupied site with a shaded circle:

<div align="center">
    <img src="images/hamiltonian_chain.png" width="400">
</div>

Note that we can also have multiple occupied sites or a superposition of such states.

Having a particle just sitting on a periodic chain like this is not particularly exciting. We want to introduce some dynamics, which we can do by adding some hopping terms. For the model here, we'll introduce three terms that hop between nearest neighbor, next-nearest-neighbour and next-next-nearest-neighbor sites, with three different coupling strengths $\omega_1, \omega_2, \omega_3$:

<div align="center">
    <img src="images/hamiltonian_schematic.png" width="600">
</div>

Note the periodicity: in this example, $\omega_3$ maps site $1$ to sites $4$ and $6$ since $1 + 3 = 4$ and $1 - 3 = -2$ and $-2 \mod 8 = 6$.

Now that we have a clear physical picture of what we want, we can express it mathematically:

$$
H = \sum_{k=0}^{N-1} \sum_{j\in\{1, 2, 3\}} \omega_j \left(\ket{k+j}\bra{k} + \ket{k -j}\bra{k}\right)
$$

One subtle point is that we cannot just have the sum terms $\ket{k+j}\bra{k}$ on their own. Rather, they must be accompanied by the difference terms $\ket{k-j}\bra{k}$ in order for the Hamiltonian to be Hermitian. A useful exercise is to convince yourself that this works as intended (hint: consider the ways in which we can manipulate $\ket{k}\bra{k + j}$ given that we have periodic boundary conditions).

## The matrix form of the Hamiltonian

This Hamiltonian is pretty simple to interpret as a matrix. The majority of the structure consists of a band-diagonal structure, with 0s on the main diagonal (our Hamiltonian has no terms that preserve the site a particle sits on) and $\omega_{k}$ as you move out to the $k$-th diagonal. The periodic structure means that these bands also "wrap" over the edges of the matrix, so we also see some additional terms farther away from the main diagonal.

Here's an example for $n = 3$ qubits (and thus $2^n = 8$ sites), with the zeros colored in light gray and the different $\omega_k$ terms colored to match the diagram above:

$$
H^{(3)} =
\begin{pmatrix}
\begin{array}{cccccccc}
{\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} \\ {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} \\ {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} \\ {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} \\ {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} \\ {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} \\ {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} \\ {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0}
\end{array}
\end{pmatrix}
$$

Applying this Hamiltonian to the state $\ket{1}$ should result in a state that mixes contributions from sites 0, 2, 3, 4, 6 and 7:

$$
\begin{align*}
H^{(3)}\ket{1} &= \sum_{k=0}^{7} \sum_{j\in\{1, 2, 3\}} \omega_j \left(\ket{k+j}\bra{k} + \ket{k -j}\bra{k}\right)\ket{1} \\
&= {\color{#AB3377}\omega_{1}}\left(\ket{2} + \ket{0}\right) + {\color{#66CCEE}\omega_{2}}\left(\ket{3} + \ket{7}\right) + {\color{#228833}\omega_{3}}\left(\ket{4} + \ket{6}\right) \ .
\end{align*}
$$

We can also verify this by direct matrix multiplication:

$$
H^{(3)}\ket{1} =
\begin{pmatrix}
\begin{array}{cccccccc}
{\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} \\ {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} \\ {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} \\ {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} \\ {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} \\ {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} \\ {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0} & {\color{#AB3377}\omega_{1}} \\ {\color{#AB3377}\omega_{1}} & {\color{#66CCEE}\omega_{2}} & {\color{#228833}\omega_{3}} & {\color{lightgray}0} & {\color{#228833}\omega_{3}} & {\color{#66CCEE}\omega_{2}} & {\color{#AB3377}\omega_{1}} & {\color{lightgray}0}
\end{array}
\end{pmatrix}
\begin{pmatrix}
{\color{lightgray}0} \\
1 \\
{\color{lightgray}0} \\
{\color{lightgray}0} \\
{\color{lightgray}0} \\
{\color{lightgray}0} \\
{\color{lightgray}0} \\
{\color{lightgray}0}
\end{pmatrix}
=
\begin{pmatrix}
{\color{#AB3377}\omega_{1}} \\
{\color{lightgray}0} \\
{\color{#AB3377}\omega_{1}} \\
{\color{#66CCEE}\omega_{2}} \\
{\color{#228833}\omega_{3}} \\
{\color{lightgray}0} \\
{\color{#228833}\omega_{3}} \\
{\color{#66CCEE}\omega_{2}}
\end{pmatrix}
$$

## Normalizing the Hamiltonian

There is one remaining thing that we need to do before we have a Hamiltonian that we are ready to block encode and qubitize. As written, the Hamiltonian is not normalized, and so the state we generated for $H^{(3)}\ket{1}$ is not physical. The physical state that we can actually realize on a quantum computer is the normalized version of this, where the normalization is with respect to the $\ell_2$-norm of the unnormalized state:

$$
\left|\left|H^{(3)}\ket{1}\right|\right|_2 := \sqrt{\sum_{k=0}^{N-1} \left|\bra{k} H^{(3)} \ket{1}\right|^2} = \sqrt{2\left(|\omega_1|^2 + |\omega_2|^2 + |\omega_3|^2\right)} \ .
$$

Our final state is therefore

$$
\frac{H^{(3)}\ket{1}}{\left|\left|H^{(3)}\ket{1}\right|\right|_2} = \frac{1}{\sqrt{2\left(|\omega_1|^2 + |\omega_2|^2 + |\omega_3|^2\right)}}\big(\omega_1\left(\ket{2} + \ket{0}\right) + \omega_2\left(\ket{3} + \ket{7}\right) + \omega_3\left(\ket{4} + \ket{6}\right) \big)
$$

## The eigenstates of the Hamiltonian

Now that we know how our chosen Hamiltonian acts on a computational basis state, we can choose a more interesting starting state. We're going to build up to performing quantum phase estimation on an encoding of this Hamiltonian, and for that we need an eigenstate. What do the eigenstates of this Hamiltonian look like?

We can answer this question with just a _little_ bit of really cool physics. This won't be for everyone, so for anybody who wants to just get to the code we'll keep this optional. Interested readers can click the expand below:

<details>

<summary><b>Deriving the eigenstates of the Hamiltonian</b></summary>

The first thing to note is that our Hamiltonian is translationally invariant by construction. This means that it commutes with the translation operator $T$, defined as

$$
T\ket{k} = \ket{k + 1}
$$

As such, $T$ and $H$ share a set of eigenvectors, so we can obtain the eigenstates for $H$ by analyzing the simpler operator $T$.

Let $\ket{\psi}$ be an eigenstate of $T$ such that

$$
T\ket{\psi} = \lambda\ket{\psi}
$$

We can expand $\ket{\psi}$ in the computational basis, with coefficients $c_k$ to be determined:

$$
\ket{\psi} = \sum_{k=0}^{N-1} c_k \ket{k}
$$

Applying $T$ in this representation yields

$$
\begin{align*}
T\ket{\psi} &= \sum_{k=0}^{N-1} c_k \ket{k + 1} \\
&= \sum_{k=0}^{N-1} c_{k-1} \ket{k} \\
&= \lambda \sum_{k=0}^{N-1} c_k \ket{k}
\end{align*}
$$

Since each of the $\ket{k}$ are linearly independent, we have that for all $k$ $c_{k-1} = \lambda c_k$, or equivalently $c_k = \lambda^{-1} c_{k-1}$.

We can thus define a recurrence relation:

$$
\begin{align*}
c_1 &= \lambda^{-1} c_0 \\
c_2 &= \lambda^{-1}\lambda^{-1}c_0 = \lambda^{-2}c_0 \\
&\vdots \\
c_k &= \lambda^{-k} c_0
\end{align*}
$$

But we know that $c_N = c_0$, since we impose periodic boundary conditions, meaning that $\lambda^{-N} = 1$. $\lambda$ is thus an $N$th root of unity:

$$
\lambda = e^{2\pi i /N} 
$$

Plugging back into the expansion of $\ket{\psi}$, we have

$$
\ket{\psi} = \sum_{k=0}^{N-1} e^{-2\pi i k/N} c_0 \ket{k}
$$

for some choice of $c_0$. Since our final state must be normalized, and since global phases are not physical in quantum systems, the only sensible choice is $c_0 = 1/\sqrt{N}$. Thus

$$
\ket{\psi} = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{-2\pi i k/N} \ket{k}
$$

Note that we could have chosen powers of $\lambda$ instead, since $1^p = 1$, meaning we have a whole family of states to chose from:

$$
\ket{p} = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{-2\pi i k p/N} \ket{k}
$$

These are precisely the momentum states for our system!

This illustrates one of the deepest and most profound ideas in physics: by starting from the idea of translational invariance, we obtained our eigenstates (i.e. the stationary states of our system) as momentum states. Translational invariance thus _necessitates_ conservation of momentum! This is an illustration of [Noether's theorem](https://en.wikipedia.org/wiki/Noether%27s_theorem), one of, if not the most important theorems in physics.

The result from the collapsed derivation above is that the eigenstates for our system are _momentum_ eigenstates:

$$
\ket{p} = \frac{1}{\sqrt{N}}\sum_{\ell=0}^{N-1} \exp\left(-\frac{2\pi i \ell p}{N}\right)\ket{\ell}
$$

Keen-eyed readers will note that this is a Fourier state and can be obtained by applying a quantum Fourier transform onto a computational basis state. (You can learn more about this transform in the [QFT kata](https://github.com/PsiQ/workbench-quantum-katas/tree/main/QFT).)

## The eigenvalues of the Hamiltonian

To see what our eigenvalue $E_p$ is for the eigenstate $\ket{p}$, we can directly compute the effect of the Hamiltonian on it:

$$
\begin{align*}
H\ket{p} &= \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} \sum_{j\in\{1, 2, 3\}} \omega_j \left(\ket{k + j}\bra{k} + \ket{k - j}\bra{k}\right) \left(\sum_{\ell=0}^{N-1} \exp \left( - \frac{2\pi i \ell p}{N} \right) \ket{\ell} \right) \\
&= \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} \sum_{j\in\{1, 2, 3\}} \omega_j \exp \left( - \frac{2\pi i k p}{N} \right) \left(\ket{k + j} + \ket{k - j}\right) \\
&= \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} \sum_{j\in\{1, 2, 3\}} \omega_j\left(\exp\left( \frac{-2\pi i (k - j) p}{N} \right) + \exp\left( \frac{-2\pi i (k + j) p}{N} \right) \right)\ket{k} \\
&= \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} \sum_{j\in\{1, 2, 3\}} \omega_j\left(\exp\left( \frac{2\pi i j p}{N} \right) + \exp\left( \frac{-2\pi i j p}{N} \right) \right) \exp \left( - \frac{2\pi i k p}{N} \right) \ket{k} \\
&= \sum_{j\in\{1, 2, 3\}} 2\omega_j\cos\left(\frac{2\pi j p}{N}\right) \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} \exp \left( - \frac{2\pi i k p}{N} \right) \ket{k} \\
&= \underbrace{\sum_{j\in\{1, 2, 3\}} 2\omega_j\cos\left(\frac{2\pi j p}{N}\right)}_{E_p}\ket{p}
\end{align*}
$$

These are the eigenstates and eigenvalues that we'll verify and then use for testing our code in the remainder of the kata.

> One final thing to note before we dive into the code is how systematic this construction is:
> -  We drew a picture of a physical system.
> -  We specified some interactions.
> -  We wrote down what the mathematics should look like to represent that system.
> 
> We had to take care of some details to ensure the construction was valid, but aside from that everything else was derived.
>
> This, and the circuit compilation that follows, is pretty much exactly how real quantum applications are derived. The systems are more complicated, there are more subtleties to catch, and of course the compilation is more complicated, but in broad strokes the steps are the same. If you have a problem in mind that you can map out in this way, you might consider going through these same steps – who knows, maybe you'll find the next leading quantum application?

## Demo: The matrix of the Hamiltonian

To start with, let's construct the matrix of the Hamiltonian for any given $\omega$ programmatically. We'll use this construction later when writing tests for our quantum program.

In [1]:
from test_QubitizedHamiltonianQPE import print_colored_matrix

def hamiltonian_matrix(n: int, w: list[float]) -> list[list[float]]:
    """Get the matrix of the Hamiltonian for n qubits for the given values of w."""
    n_sites = 2 ** n
    matrix = []
    for k in range(n_sites):
        row = [0.0] * n_sites
        for j in [1, 2, 3]:
            row[(n_sites + k + j) % n_sites] = row[(n_sites + k - j) % n_sites] = w[j - 1]
        matrix.append(row)
    return matrix

n = 3
w = [0.5, 0.3, 0.2]
ham_matrix = hamiltonian_matrix(n, w)
print_colored_matrix(ham_matrix, w)

0.0,0.5,0.3,0.2,0.0,0.2,0.3,0.5
0.5,0.0,0.5,0.3,0.2,0.0,0.2,0.3
0.3,0.5,0.0,0.5,0.3,0.2,0.0,0.2
0.2,0.3,0.5,0.0,0.5,0.3,0.2,0.0
0.0,0.2,0.3,0.5,0.0,0.5,0.3,0.2
0.2,0.0,0.2,0.3,0.5,0.0,0.5,0.3
0.3,0.2,0.0,0.2,0.3,0.5,0.0,0.5
0.5,0.3,0.2,0.0,0.2,0.3,0.5,0.0


The matrix printed by the `print_colored_matrix` method is color-coded to match the diagram and the formulas above.

## Demo: The eigenvalues of the Hamiltonian

Now, let's calculate the eigenvalues and the eigenstates of the matrix we just constructed using the formulas we derived above and to check that they indeed behave as eigenvalues and eigenstates. Later, we will use these eigenvalues for testing.

In [2]:
import numpy as np

# Check that the matrix is Hermitian: compare it with its conjugate transpose
ham = np.array(ham_matrix)
is_hermitian = np.allclose(ham, ham.conjugate().T)
print(f"The matrix is {'Hermitian' if is_hermitian else 'NOT Hermitian'}")


def pretty_print(eigenvalue, eigenvector, index):
    """Helper function to print formatted eigenvalue-eigenvector pair with a given index."""
    print(f"E_{index} = {round(eigenvalue, 6):<10} "
          f"|{index}⟩ = {[round(float(v), 4) for v in eigenvector]}")

# Evaluate the formula above for eigenvalues and eigenvectors
from cmath import cos, exp, pi, sqrt
print("Eigenvalues and eigenstates for each |p⟩")
n_sites = 2 ** n
for p in range(n_sites):
    e_p = sum([2 * w[j - 1] * cos(2 * pi * j * p / n_sites) for j in [1, 2, 3]]).real
    ket_p = [(1 / sqrt(n_sites) * exp(- 2 * pi * 1j * k * p / n_sites)).real for k in range(n_sites)]
    pretty_print(e_p, ket_p, p)
    # Check that applying the Hamiltonian to the |p⟩ indeed gives us Eₚ|p⟩
    assert np.allclose(ham @ np.array(ket_p), e_p * np.array(ket_p))

The matrix is Hermitian
Eigenvalues and eigenstates for each |p⟩
E_0 = 2.0        |0⟩ = [0.3536, 0.3536, 0.3536, 0.3536, 0.3536, 0.3536, 0.3536, 0.3536]
E_1 = 0.424264   |1⟩ = [0.3536, 0.25, 0.0, -0.25, -0.3536, -0.25, -0.0, 0.25]
E_2 = -0.6       |2⟩ = [0.3536, 0.0, -0.3536, -0.0, 0.3536, 0.0, -0.3536, -0.0]
E_3 = -0.424264  |3⟩ = [0.3536, -0.25, -0.0, 0.25, -0.3536, 0.25, 0.0, -0.25]
E_4 = -0.8       |4⟩ = [0.3536, -0.3536, 0.3536, -0.3536, 0.3536, -0.3536, 0.3536, -0.3536]
E_5 = -0.424264  |5⟩ = [0.3536, -0.25, 0.0, 0.25, -0.3536, 0.25, -0.0, -0.25]
E_6 = -0.6       |6⟩ = [0.3536, -0.0, -0.3536, 0.0, 0.3536, -0.0, -0.3536, -0.0]
E_7 = 0.424264   |7⟩ = [0.3536, 0.25, -0.0, -0.25, -0.3536, -0.25, -0.0, 0.25]


## Part I conclusion

In the first part of this kata, you've defined the Hamiltonian we'll be using throughout the next parts and learned to construct its matrix form and its eigenvalues and eigenstates. In [part II](./QubitizedHamiltonianQPE_2_BlockEncoding.ipynb), you will learn to build the block encoding of this Hamiltonian.